# Результаты экспериментов (exp3)

В этом ноутбуке используется код из `experiments/exp3` (агенты, среда, визуализация).

Где 0 = сотрудничать, 1 = предать.

In [ ]:
# Настраиваем sys.path, чтобы импортировать модули из experiments/exp3
import os, sys
repo_root = os.path.abspath(os.path.join(os.getcwd()))
exp3_path = os.path.join(repo_root, 'experiments', 'exp3')
if exp3_path not in sys.path:
    sys.path.insert(0, exp3_path)
print('📦 sys.path настроен для exp3:', exp3_path)

In [ ]:
# Импорты и полезные настройки
import math
import random
import numpy as np
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x):
        return x

# Импортируем реализации из exp3
from bots import SmartAgent, SimpleBot
from environment import Game, GameFactory
from viz_utils import smooth, plot_rewards_and_coop

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
print('✅ Библиотеки и модули загружены')

In [ ]:
# Эксперимент 1: Умный агент против всегда предающего (2 игрока)
smart_agent = SmartAgent('Умник')
opponent = SimpleBot('betray')
pd_game = GameFactory.create_generalized_prisoners_dilemma(2)

rewards = []
actions = []
print('🎯 Начинаем обучение против бота-предателя (2 игрока)...')
for i in tqdm(range(1000)):
    a = smart_agent.choose_action()
    b = opponent.choose_action()
    r1 = float(pd_game.get_payoff([a, b]))
    smart_agent.learn(a, r1)
    rewards.append(r1)
    actions.append(a)
    if i % 100 == 0 and i > 0:
        smart_agent.exploration = max(0.01, smart_agent.exploration * 0.9)

print('✅ Обучение завершено!')
smart_agent.print_stats()
plot_rewards_and_coop(rewards, actions, title_prefix=smart_agent.name)

In [ ]:
# Эксперимент 2: Два умных агента (2 игрока)
agent1 = SmartAgent('Алиса')
agent2 = SmartAgent('Боб')
pd_game = GameFactory.create_generalized_prisoners_dilemma(2)

rewards1, rewards2 = [], []
actions1, actions2 = [], []
print('🧠 Запускаем игру двух умных агентов (2 игрока)...')
for i in tqdm(range(2000)):
    a1 = agent1.choose_action()
    a2 = agent2.choose_action()
    r1 = float(pd_game.get_payoff([a1, a2]))
    r2 = float(pd_game.get_payoff([a2, a1]))
    agent1.learn(a1, r1)
    agent2.learn(a2, r2)
    rewards1.append(r1); rewards2.append(r2)
    actions1.append(a1); actions2.append(a2)
    if i % 200 == 0 and i > 0:
        agent1.exploration = max(0.01, agent1.exploration * 0.9)
        agent2.exploration = max(0.01, agent2.exploration * 0.9)

agent1.print_stats()
agent2.print_stats()

# Графики для обоих агентов
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
window = 50
axes[0,0].plot(smooth(rewards1, window), color='blue'); axes[0,0].set_title('Награды Алисы')
axes[0,1].plot(smooth(rewards2, window), color='red');  axes[0,1].set_title('Награды Боба')
coop1 = [1 - np.mean(actions1[i:i+window]) for i in range(max(1, len(actions1)-window))]
coop2 = [1 - np.mean(actions2[i:i+window]) for i in range(max(1, len(actions2)-window))]
axes[1,0].plot(coop1, color='blue'); axes[1,0].set_title('Сотрудничество Алисы'); axes[1,0].set_ylim(0,1)
axes[1,1].plot(coop2, color='red');  axes[1,1].set_title('Сотрудничество Боба');  axes[1,1].set_ylim(0,1)
plt.tight_layout(); plt.show()

In [ ]:
# Эксперимент 3a: Три игрока — умник против двух предателей
smart_agent = SmartAgent('Умник')
opponent_1 = SimpleBot('betray')
opponent_2 = SimpleBot('betray')
tri_game = GameFactory.create_custom_three_player_game()

rewards = []
actions = []
print('🎯 Три игрока: умник против двух предателей...')
for i in tqdm(range(1000)):
    a = smart_agent.choose_action()
    b = opponent_1.choose_action()
    c = opponent_2.choose_action()
    r1 = float(tri_game.get_payoff([a, b, c]))
    smart_agent.learn(a, r1)
    rewards.append(r1); actions.append(a)
    if i % 100 == 0 and i > 0:
        smart_agent.exploration = max(0.01, smart_agent.exploration * 0.9)

smart_agent.print_stats()
plot_rewards_and_coop(rewards, actions, title_prefix=smart_agent.name)

In [ ]:
# Эксперимент 3b: Три игрока — умник против предателя и кооператора
smart_agent = SmartAgent('Умник')
opponent_1 = SimpleBot('betray')
opponent_2 = SimpleBot('cooperate')
tri_game = GameFactory.create_custom_three_player_game()

rewards = []
actions = []
print('🎯 Три игрока: умник против предателя и кооператора...')
for i in tqdm(range(1000)):
    a = smart_agent.choose_action()
    b = opponent_1.choose_action()
    c = opponent_2.choose_action()
    r1 = float(tri_game.get_payoff([a, b, c]))
    smart_agent.learn(a, r1)
    rewards.append(r1); actions.append(a)
    if i % 100 == 0 and i > 0:
        smart_agent.exploration = max(0.01, smart_agent.exploration * 0.9)

smart_agent.print_stats()
plot_rewards_and_coop(rewards, actions, title_prefix=smart_agent.name)

In [ ]:
# Эксперимент 4: Три умных агента
agent1 = SmartAgent('Алиса')
agent2 = SmartAgent('Боб')
agent3 = SmartAgent('Чарли')
tri_game = GameFactory.create_custom_three_player_game()

rewards1, rewards2, rewards3 = [], [], []
actions1, actions2, actions3 = [], [], []
print('🧠 Запускаем игру трёх умных агентов...')
for i in tqdm(range(2000)):
    a1 = agent1.choose_action()
    a2 = agent2.choose_action()
    a3 = agent3.choose_action()
    r1 = float(tri_game.get_payoff([a1, a2, a3]))
    r2 = float(tri_game.get_payoff([a2, a3, a1]))
    r3 = float(tri_game.get_payoff([a3, a1, a2]))
    agent1.learn(a1, r1)
    agent2.learn(a2, r2)
    agent3.learn(a3, r3)
    rewards1.append(r1); rewards2.append(r2); rewards3.append(r3)
    actions1.append(a1); actions2.append(a2); actions3.append(a3)
    if i % 200 == 0 and i > 0:
        agent1.exploration = max(0.01, agent1.exploration * 0.9)
        agent2.exploration = max(0.01, agent2.exploration * 0.9)
        agent3.exploration = max(0.01, agent3.exploration * 0.9)

agent1.print_stats(); agent2.print_stats(); agent3.print_stats()

# Большой график с результатами
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 2, figsize=(15, 10))
window = 50
axes[0,1].plot(smooth(rewards1, window), color='blue');  axes[0,1].set_title('Награды Алисы')
axes[1,1].plot(smooth(rewards2, window), color='red');   axes[1,1].set_title('Награды Боба')
axes[2,1].plot(smooth(rewards3, window), color='green'); axes[2,1].set_title('Награды Чарли')
coop1 = [1 - np.mean(actions1[i:i+window]) for i in range(max(1, len(actions1)-window))]
coop2 = [1 - np.mean(actions2[i:i+window]) for i in range(max(1, len(actions2)-window))]
coop3 = [1 - np.mean(actions3[i:i+window]) for i in range(max(1, len(actions3)-window))]
axes[0,0].plot(coop1, color='blue');  axes[0,0].set_title('Сотрудничество Алисы');  axes[0,0].set_ylim(0,1)
axes[1,0].plot(coop2, color='red');   axes[1,0].set_title('Сотрудничество Боба');   axes[1,0].set_ylim(0,1)
axes[2,0].plot(coop3, color='green'); axes[2,0].set_title('Сотрудничество Чарли'); axes[2,0].set_ylim(0,1)
plt.tight_layout(); plt.show()